In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "rock"
xdmf_file_name = "rock.xdmf"

load_params = ABS_PATH + "files/pred_parameters.csv"

dbdir = ABS_PATH + "trainPOD/"

In [ ]:
import numpy as np
import fenics as fe
import pandas as pd

from modelaquisition.msh2xdmf import Msh2Xdmf

from fem_problems.invTP1.finite_element import PoissonFEM
from fem_problems.invTP1.rbnics_pod import PODReduction

In [ ]:
df = pd.read_csv(load_params, sep=";", index_col=0)

mu_pred = [df.iloc[i, 0] for i in range(3)]

In [ ]:
# Path to the XDMF file
file_path = ABS_PATH + xdmf_file_name

# Carica la mesh da file XDMF
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

In [ ]:
mu_range = [
    (0, 1.),
    (0, 1.),
    (0, 1.),
]

In [ ]:
fem_p = PoissonFEM(mesh)
pod = PODReduction(mu_range, fem_p)

In [ ]:
pod.load_reduction(directory=dbdir, filename=model_name)

In [ ]:
mu_test = [.1, .2, .5]
mat, _ = pod.solve_online_rom(mu_test, pod.num_basis)
fom_m, _ = pod.fem_p.solve_online_fem(mu_test)
real_m, _ = pod.fem_p.exact_solution_online(mu_test)

print("ERRORE FOM-ESATTA")
e, f = pod.fem_p.compute_error(fom_m, real_m)
print("ERRORE ROM-FOM")
a, b = pod.fem_p.compute_error(mat, fom_m)
print("ERRORE ROM-ESATTA")
c, d = pod.fem_p.compute_error(mat, real_m)

In [ ]:
rock = Msh2Xdmf(ABS_PATH + xdmf_file_name, model_name)

rock.reset_files("_sol_rom")
rock.reset_files("_sol_fom")
rock.reset_files("_sol_an")

rock.add_solution(mat, "_sol_rom", "solution_rom")
rock.add_solution(fom_m, "_sol_fom", "solution_fom")
rock.add_solution(real_m, "_sol_an", "solution_an")

In [ ]:
err = np.abs(mat - real_m) / np.linalg.norm(real_m)

rock.reset_files("_err_rom_an")
rock.add_solution(err, "_err_rom_an", "error")

In [ ]:
err_ass_fom_rom = np.abs(mat - fom_m)
err_rel_fom_rom = err_ass_fom_rom / np.linalg.norm(fom_m)

rock.reset_files("_err_fom_rom")
rock.reset_files("_err_rel_fom_rom")
rock.add_solution(err_ass_fom_rom, "_err_fom_rom", "error_fom_rom")
rock.add_solution(err_rel_fom_rom, "_err_rel_fom_rom", "error_rel_fom_rom")

In [ ]:
err_fom_an = np.abs(fom_m - real_m)
err_rel_fom_an = err_fom_an / np.linalg.norm(real_m)

rock.reset_files("_err_fom")
rock.add_solution(err_rel_fom_an, "_err_fom", "error_fom")